# osu! MapForge – Cloud-Modus (experimentell)
KI-Maps mit **Mapperatorinator** auf einer kostenlosen Google-GPU erzeugen – für alle ohne NVIDIA-Karte.

1. Oben **Laufzeit → Laufzeittyp ändern → T4-GPU** wählen.
2. Die Zellen der Reihe nach ausführen (▶). Zelle 1 dauert beim ersten Mal ca. 3–5 Minuten.
3. In Zelle 3 die Einstellungen aus MapForge einfügen (Knopf „Cloud (Colab)“ im Generator kopiert sie in die Zwischenablage).
4. Am Ende wird eine `.osu` heruntergeladen → in MapForge im Feinschliff mit **„+ Stufe / .osz …“** dazunehmen (gleiche Audiodatei).

Hinweis: Mit KI erstellte Maps dürfen laut osu!-Regeln nicht ranked werden.

In [ ]:
# 1 · Mapperatorinator einrichten (einmal pro Sitzung)
!nvidia-smi -L || echo "Keine GPU – Laufzeit → Laufzeittyp ändern → T4-GPU"
!git clone --depth 1 https://github.com/OliBomby/Mapperatorinator.git /content/Mapperatorinator 2>/dev/null || (cd /content/Mapperatorinator && git pull --ff-only)
%cd /content/Mapperatorinator
!pip -q install -r requirements.txt
print("Fertig.")

In [ ]:
# 2 · Audiodatei hochladen (MP3/OGG/WAV/FLAC/M4A)
import os
from google.colab import files
up = files.upload()
AUDIO = os.path.abspath(list(up.keys())[0])
print("Audio:", AUDIO)

In [ ]:
# 3 · Einstellungen – hier den Block aus MapForge („Cloud (Colab)“) einfügen oder von Hand anpassen
SETTINGS = {
    "difficulty": 5.0, "circle_size": 4, "approach_rate": 9, "overall_difficulty": 8, "hp_drain_rate": 5,
    "slider_multiplier": 1.4, "version": "KI", "title": "", "artist": "", "creator": "",
    "mapper_id": None, "year": 2023, "descriptors": [], "negative_descriptors": [], "cfg_scale": 1.0,
    "temperature": 0.85, "seed": 1, "super_timing": False, "hitsounded": True,
}

In [ ]:
# 4 · Map erzeugen
import shlex, subprocess, glob, os, json
OUT = "/content/out"
os.makedirs(OUT, exist_ok=True)
def q(v):
    s = str(v).replace("\\", "/").replace("'", "\\'")
    return "'" + s + "'"
args = ["python", "inference.py", f"audio_path={q(AUDIO)}", f"output_path={q(OUT)}", "gamemode=0"]
for k, v in SETTINGS.items():
    if v is None or v == [] : continue
    if isinstance(v, bool): v = "true" if v else "false"
    elif isinstance(v, list): v = "[" + ",".join(q(x) for x in v) + "]"
    elif isinstance(v, str): v = q(v)
    args.append(f"{k}={v}")
print(" ".join(args))
subprocess.run(args, check=True)
res = sorted(glob.glob(OUT + "/*.osu"), key=os.path.getmtime)
print("Erzeugt:", res[-1] if res else "nichts")

In [ ]:
# 5 · Herunterladen → in MapForge im Feinschliff mit „+ Stufe / .osz …“ dazunehmen
from google.colab import files
files.download(res[-1])